In [ ]:
# ================================================================
# TELE-SAP ANALYSIS — Q4 NOTEBOOK — UNIFIED FINAL VERSION
# Compact figures + fixed legend caption for Colab
# ================================================================

# If needed, uncomment:
# !pip install -q pandas numpy plotly

import warnings
import numpy as np
import pandas as pd
import plotly.express as px

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------
# FILE PATH IN COLAB
# ---------------------------------------------------------------
file_path = "TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv"


def load_csv() -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    return df


# ---------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------
def shorten_label(text: str, max_len: int = 24) -> str:
    text = str(text)
    return text if len(text) <= max_len else text[: max_len - 3] + "..."


def compact_height(
    n_items: int,
    base: int = 260,
    step: int = 34,
    min_h: int = 320,
    max_h: int = 700,
) -> int:
    h = base + n_items * step
    return max(min_h, min(h, max_h))


def categorize_notifiable_disease_expanded(column_name: str):
    col_lower = column_name.lower()

    forbidden_terms = [
        "exame", "teste", "sorologia", "rastreio", "solicita",
        "vacina", "suspeita", "pedido"
    ]
    if any(term in col_lower for term in forbidden_terms):
        return None

    if any(k in col_lower for k in ["hiv", "aids", "sida"]):
        return "HIV/AIDS"
    if any(k in col_lower for k in ["sifilis", "sífilis"]):
        return "Syphilis"
    if "tuberculose" in col_lower:
        return "Tuberculosis"
    if "hepatite" in col_lower:
        return "Hepatitis"
    if "dengue" in col_lower:
        return "Dengue"
    if "hanseníase" in col_lower:
        return "Leprosy"
    if "covid" in col_lower:
        return "COVID-19"
    if "meningite" in col_lower:
        return "Meningitis"
    if "malária" in col_lower:
        return "Malaria"
    if any(k in col_lower for k in ["varíola", "mpox", "monkeypox"]):
        return "Mpox/Smallpox"
    if any(k in col_lower for k in ["zika", "chikungunya"]):
        return "Arboviruses (Zika/Chikungunya)"
    if "leishmaniose" in col_lower:
        return "Leishmaniasis"
    if "chagas" in col_lower:
        return "Chagas Disease"
    if "leptospirose" in col_lower:
        return "Leptospirosis"
    if "esquistossomose" in col_lower:
        return "Schistosomiasis"
    if "toxoplasmose" in col_lower:
        return "Toxoplasmosis"
    if "raiva" in col_lower or "mordedura" in col_lower or "animal" in col_lower:
        return "Anti-Rabies Care"
    if "violência" in col_lower or "violencia" in col_lower or "suicídio" in col_lower:
        return "Violence / Self-harm"
    if "intoxicação" in col_lower or "intoxicacao" in col_lower or "veneno" in col_lower:
        return "Exogenous Intoxication"
    if "peçonhento" in col_lower or "escorpião" in col_lower or "cobra" in col_lower:
        return "Venomous Animal Accident"

    return None


def build_disease_map(df: pd.DataFrame):
    disease_map = {}
    for col in df.columns:
        category = categorize_notifiable_disease_expanded(col)
        if category:
            disease_map[col] = category
    detected_diseases = sorted(list(set(disease_map.values())))
    return disease_map, detected_diseases


def get_positive_values():
    return ["checked", "sim", "yes", "verdadeiro", "true", "1"]


def apply_compact_bar_layout(fig, title_text: str, n_items: int):
    fig.update_layout(
        barmode="stack",
        height=compact_height(n_items),
        title=dict(
            text=title_text,
            font=dict(size=17),
            x=0.01,
        ),
        xaxis=dict(
            title="Percentage of cases",
            range=[0, 100],
            showgrid=False,
            tickfont=dict(size=11),
            title_font=dict(size=12),
        ),
        yaxis=dict(
            title="",
            tickfont=dict(size=11),
            automargin=True,
            showgrid=False,
            zeroline=False,
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5,
            font=dict(size=11),
            title_text="",
            bgcolor="rgba(255,255,255,0.85)",
        ),
        margin=dict(l=90, r=40, t=110, b=40),
        plot_bgcolor="white",
        paper_bgcolor="white",
        bargap=0.28,
    )

    fig.update_traces(
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(color="white", size=10),
        marker_line_width=0,
    )


# ---------------------------------------------------------------
# FIGURE 1 — SINAN audit
# ---------------------------------------------------------------
def figure_1_sinan_audit(df: pd.DataFrame) -> None:
    cols_sinan = [c for c in df.columns if "sinan" in c.lower() or "notifica" in c.lower()]
    if not cols_sinan:
        raise ValueError("SINAN column was not found.")
    col_sinan = cols_sinan[0]

    disease_map, detected_diseases = build_disease_map(df)

    empty_values = ["nan", "", "não", "nao", "none", "false", "0"]
    df = df.copy()
    df["Has_SINAN"] = ~df[col_sinan].astype(str).str.strip().str.lower().isin(empty_values)

    positive_values = get_positive_values()

    for disease in detected_diseases:
        disease_cols = [c for c, d in disease_map.items() if d == disease]
        df[disease] = df[disease_cols].apply(
            lambda row: row.astype(str).str.strip().str.lower().isin(positive_values).any(),
            axis=1,
        )

    agg_dict = {disease: "max" for disease in detected_diseases}
    agg_dict["Has_SINAN"] = "max"
    df_patients = df.groupby("Record ID").agg(agg_dict).reset_index()

    df_melted = df_patients.melt(
        id_vars=["Record ID", "Has_SINAN"],
        value_vars=detected_diseases,
        var_name="Disease",
        value_name="Has_Disease",
    )

    df_audit = df_melted[df_melted["Has_Disease"] == True].copy()
    df_audit["SINAN_Status"] = np.where(df_audit["Has_SINAN"], "Recorded", "Missing")

    if df_audit.empty:
        print("No confirmed notifiable disease was found in this dataset.")
        return

    df_grouped = df_audit.groupby(["Disease", "SINAN_Status"]).size().reset_index(name="Cases")
    df_totals = df_audit.groupby("Disease").size().reset_index(name="Total")
    df_grouped = pd.merge(df_grouped, df_totals, on="Disease")
    df_grouped["Percentage"] = (df_grouped["Cases"] / df_grouped["Total"]) * 100

    order = df_totals.sort_values("Total", ascending=True)["Disease"].tolist()
    short_map = {d: shorten_label(d, 24) for d in order}
    df_grouped["Disease_Short"] = df_grouped["Disease"].map(short_map)
    ordered_short = [short_map[d] for d in order]

    fig = px.bar(
        df_grouped,
        y="Disease_Short",
        x="Percentage",
        color="SINAN_Status",
        orientation="h",
        category_orders={"Disease_Short": ordered_short},
        color_discrete_map={
            "Missing": "#d9534f",
            "Recorded": "#5cb85c",
        },
        custom_data=["Disease", "Cases", "Total", "Percentage"],
        text=df_grouped["Percentage"].map(lambda v: f"{v:.1f}%"),
    )

    fig.update_traces(
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Cases: %{customdata[1]}<br>"
            "Total: %{customdata[2]}<br>"
            "Share: %{customdata[3]:.1f}%<extra></extra>"
        )
    )

    apply_compact_bar_layout(fig, f"SINAN Audit - Total: {len(df_audit)} Cases", len(df_totals))
    fig.show()

    df_table = df_grouped.copy()
    df_table["Percentage"] = df_table["Percentage"].round(1)
    df_table = df_table.rename(
        columns={
            "Disease": "Condition / Disease",
            "SINAN_Status": "Notification Status",
            "Cases": "Number of Patients",
            "Total": "Total Disease Cases",
            "Percentage": "Proportion (%)",
        }
    )[["Condition / Disease", "Notification Status", "Number of Patients", "Total Disease Cases", "Proportion (%)"]]

    output_file = "Expanded_SINAN_Audit_Table_EN.csv"
    df_table.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\n[SUCCESS] Chart generated. Table saved as: '{output_file}'.\n")


# ---------------------------------------------------------------
# FIGURE 2 — Infectious diseases vs Ophthalmology request
# ---------------------------------------------------------------
def figure_2_ophthalmology_vs_infectious_diseases(df: pd.DataFrame) -> None:
    disease_map, detected_diseases = build_disease_map(df)
    cols_ophthalmo = [c for c in df.columns if "oftalmo" in c.lower()]

    if not cols_ophthalmo:
        print("No Ophthalmology-related column was found.")
        return

    positive_values = get_positive_values()
    df = df.copy()

    df["Has_Ophthalmology"] = df[cols_ophthalmo].apply(
        lambda row: row.astype(str).str.strip().str.lower().isin(positive_values).any(),
        axis=1,
    )

    for disease in detected_diseases:
        disease_cols = [c for c, d in disease_map.items() if d == disease]
        df[disease] = df[disease_cols].apply(
            lambda row: row.astype(str).str.strip().str.lower().isin(positive_values).any(),
            axis=1,
        )

    agg_dict = {disease: "max" for disease in detected_diseases}
    agg_dict["Has_Ophthalmology"] = "max"
    df_patients = df.groupby("Record ID").agg(agg_dict).reset_index()

    df_melted = df_patients.melt(
        id_vars=["Record ID", "Has_Ophthalmology"],
        value_vars=detected_diseases,
        var_name="Disease",
        value_name="Has_Disease",
    )

    df_analysis = df_melted[df_melted["Has_Disease"] == True].copy()
    df_analysis["Referral_Status"] = np.where(
        df_analysis["Has_Ophthalmology"],
        "Requested",
        "Not Requested",
    )

    if df_analysis.empty:
        print("No confirmed notifiable disease was found in this dataset.")
        return

    df_grouped = df_analysis.groupby(["Disease", "Referral_Status"]).size().reset_index(name="Cases")
    df_totals = df_analysis.groupby("Disease").size().reset_index(name="Total")
    df_grouped = pd.merge(df_grouped, df_totals, on="Total" if False else "Disease")
    df_grouped["Percentage"] = (df_grouped["Cases"] / df_grouped["Total"]) * 100

    order = df_totals.sort_values("Total", ascending=True)["Disease"].tolist()
    short_map = {d: shorten_label(d, 24) for d in order}
    df_grouped["Disease_Short"] = df_grouped["Disease"].map(short_map)
    ordered_short = [short_map[d] for d in order]

    fig = px.bar(
        df_grouped,
        y="Disease_Short",
        x="Percentage",
        color="Referral_Status",
        orientation="h",
        category_orders={"Disease_Short": ordered_short},
        color_discrete_map={
            "Requested": "#27ae60",
            "Not Requested": "#bdc3c7",
        },
        custom_data=["Disease", "Cases", "Total", "Percentage"],
        text=df_grouped["Percentage"].map(lambda v: f"{v:.1f}%"),
    )

    fig.update_traces(
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Cases: %{customdata[1]}<br>"
            "Total: %{customdata[2]}<br>"
            "Share: %{customdata[3]:.1f}%<extra></extra>"
        )
    )

    apply_compact_bar_layout(fig, "Ophthalmology Requests by Infectious Disease", len(df_totals))
    fig.show()

    df_table = df_grouped.copy()
    df_table["Percentage"] = df_table["Percentage"].round(1)
    df_table = df_table.rename(
        columns={
            "Disease": "Infectious Disease",
            "Referral_Status": "Ocular Evaluation Status",
            "Cases": "Number of Patients",
            "Total": "Total Disease Cases",
            "Percentage": "Proportion (%)",
        }
    )[["Infectious Disease", "Ocular Evaluation Status", "Number of Patients", "Total Disease Cases", "Proportion (%)"]]

    output_file = "Ophthalmology_Infectious_Diseases_Table_EN.csv"
    df_table.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"\n[SUCCESS] Chart generated tracking {len(df_analysis)} confirmed cases.")
    print(f"Base table saved as: '{output_file}'.\n")


# ---------------------------------------------------------------
# FIGURE 3 — Management summary for SINAN completeness
# ---------------------------------------------------------------
def figure_3_sinan_management_summary(df: pd.DataFrame) -> None:
    cols_sinan = [c for c in df.columns if "sinan" in c.lower() or "notifica" in c.lower()]
    if not cols_sinan:
        raise ValueError("SINAN column was not found.")
    col_sinan = cols_sinan[0]

    disease_map, detected_diseases = build_disease_map(df)
    positive_values = get_positive_values()

    df = df.copy()

    for disease in detected_diseases:
        disease_cols = [c for c, d in disease_map.items() if d == disease]
        df[disease] = df[disease_cols].apply(
            lambda row: row.astype(str).str.strip().str.lower().isin(positive_values).any(),
            axis=1,
        )

    empty_values = ["nan", "", "não", "nao", "none", "false", "0"]
    df["Has_SINAN"] = ~df[col_sinan].astype(str).str.strip().str.lower().isin(empty_values)

    agg_dict = {disease: "max" for disease in detected_diseases}
    agg_dict["Has_SINAN"] = "max"
    df_patients = df.groupby("Record ID").agg(agg_dict).reset_index()

    df_melted = df_patients.melt(
        id_vars=["Record ID", "Has_SINAN"],
        value_vars=detected_diseases,
        var_name="Disease",
        value_name="Has_Disease",
    )

    df_audit = df_melted[df_melted["Has_Disease"] == True].copy()

    total_cases = len(df_audit)
    cases_with_sinan = int(df_audit["Has_SINAN"].sum())
    blank_cases = total_cases - cases_with_sinan
    effectiveness_rate = (cases_with_sinan / total_cases) * 100 if total_cases > 0 else 0.0

    print("=" * 80)
    print("STATISTICAL SUMMARY: EPIDEMIOLOGICAL SURVEILLANCE (SINAN)")
    print("=" * 80)
    print(f"Total diagnosed cases with compulsory notification: {total_cases}")
    print(f"-> Cases with SINAN number generated/recorded: {cases_with_sinan}")
    print(f"-> Cases with blank SINAN number:             {blank_cases}")
    print(f"\nRegistration Effectiveness Rate: {effectiveness_rate:.2f}%")
    print("=" * 80)

    if effectiveness_rate < 80:
        print("WARNING: Completion rate is below the desired level.")
        print("Consider mandatory validation before record closure.")


# ---------------------------------------------------------------
# RUN EVERYTHING
# ---------------------------------------------------------------
def main():
    df = load_csv()
    figure_1_sinan_audit(df)
    figure_2_ophthalmology_vs_infectious_diseases(df)
    figure_3_sinan_management_summary(df)


main()